In [114]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.expand_frame_repr', False) # disable wrapping

In [115]:
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

def get_ext_price(symbol: str, include_extended_hours: bool = True) -> float:
	ticker = yf.Ticker(symbol)

	if include_extended_hours:
		info = ticker.info or {}
		# Prefer explicit post-market and pre-market quotes when available.
		post_market_price = info.get("postMarketPrice")
		if post_market_price is not None:
			return round(float(post_market_price), 2)

		pre_market_price = info.get("preMarketPrice")
		if pre_market_price is not None:
			return round(float(pre_market_price), 2)

		# Fallback for extended hours: latest intraday trade including pre/post data.
		intraday = ticker.history(period="1d", interval="1m", prepost=True)
		if not intraday.empty:
			close_series = intraday["Close"].dropna()
			if not close_series.empty:
				return round(float(close_series.iloc[-1]), 2)

	# Try the most up-to-date value first.
	fast_info = ticker.fast_info or {}
	last_price = fast_info.get("lastPrice")
	if last_price is not None:
		return round(float(last_price), 2)

	# Fallback: use latest close from recent history.
	history = ticker.history(period="5d")
	if history.empty:
		raise RuntimeError(f"Unable to fetch {symbol} price from yfinance")
	return round(float(history["Close"].iloc[-1]), 2)

def get_prices(tickers, include_extended_hours: bool = True) -> dict:
	"""Fetch prices for unique tickers concurrently; failures map to NaN."""
	def safe_get(symbol):
		try:
			return get_ext_price(symbol, include_extended_hours)
		except Exception as e:
			print(symbol, e)
			return float('nan')
	unique = list(dict.fromkeys(tickers))
	with ThreadPoolExecutor(max_workers=16) as ex:
		prices = list(ex.map(safe_get, unique))
	return dict(zip(unique, prices))
#print(get_ext_price("AAPL")) #testing   

In [116]:
df = pd.read_csv(r'..\data_save\out_fidelity.csv', on_bad_lines='skip')
IS_OPTION = r'^[A-Za-z.]+\d{6}[CP][\d.]+$' # option symbols look like AGQ260918C74: ticker, expiry, C/P, strike
df = df[df['Symbol'].notna() & ~df['Symbol'].astype(str).str.match(IS_OPTION)] # stocks only
df = df[df['OptionCnt'].notna()] 
print(df[['Symbol', 'OptionCnt', 'sell', 'CostBasisPerShare','buy']]) #testing

df['o_CurPrice'] = df['Symbol'].map(get_prices(df['Symbol'], include_extended_hours=False)) # fast_info only, each unique ticker once, in parallel
df['o_ref_price'] = df['sell'].combine_first(pd.to_numeric(df['CostBasisPerShare'], errors='coerce').abs())

df['o_ref_price'] = pd.to_numeric(df['o_ref_price'], errors='coerce')
df['o_Diff'] = df['o_CurPrice'] - df['o_ref_price']

df = df.sort_values('o_Diff', ascending=False)
df['sell'] = df['sell'].fillna('')
df['OptionCnt'] = pd.to_numeric(df['OptionCnt'], errors='coerce').apply(lambda x: int(x) if pd.notna(x) else '') # blank when missing or not a number
print(df[['Account', 'Symbol', 'CostBasisPerShare', 'sell', 'o_CurPrice', 'o_Diff', 'OptionCnt']])

    Symbol  OptionCnt  sell  CostBasisPerShare         buy
0     EROC        1.0  21.5                NaN         IPO
3      AGQ        5.0   NaN             101.79         NaN
28    COIN        3.0   270             318.40         NaN
39    COST        1.0   NaN            -979.55         NaN
41    CRCL        2.0    94            -182.66         NaN
42    CRCL        2.0   NaN            -143.00         NaN
53     DJT        2.0    11             -24.79          17
57    ETHE        1.0    21             -26.70         NaN
59     GLD        3.0   440                NaN         NaN
62    IONQ        1.0   NaN             -43.00         NaN
67     MNY        1.0   2.5             -14.15         NaN
68    MSTR        1.0   143            -228.31         NaN
75    NNOX        1.0   3.5             -45.70         NaN
76    NNOX        4.0   3.5             -15.83         NaN
77    NNOX        1.0   3.5             -13.60         NaN
78    NNOX        3.0   3.1              -9.21         N

In [117]:
import datetime as dt
import re
from datetime import date
from concurrent.futures import ThreadPoolExecutor
import yfinance as yf

def calc_rate(cost, symbol_to):
    if not symbol_to.strip():
        return
    symbolOp = re.split(r'([\d.]+)', symbol_to)
    final_value = float(symbolOp[3])
    final_date = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (final_date - date.today()).days+1
    rate_of_gain = float(cost) / final_value * (365 / days_hold) * 100
    return round(rate_of_gain)

def _fetch_chains(keys):
    """Fetch option chains for unique (ticker, expiry) pairs concurrently."""
    def fetch(key):
        try:
            return key, yf.Ticker(key[0]).option_chain(key[1])
        except Exception as e:
            print(key[0], e)
            return key, None
    with ThreadPoolExecutor(max_workers=16) as ex:
        return dict(ex.map(fetch, keys))

def _scan(df, opt_type):
    # Parse all symbols first so each (ticker, expiry) chain is downloaded only once.
    parsed = {}
    for i in df.index:
        try:
            symbolOp = re.split(r'([\d.]+)', df['Symbol'][i])
            myDate = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
            parsed[i] = (symbolOp[0], myDate.strftime('%Y-%m-%d'), float(symbolOp[3]))
        except Exception as e:
            print(df['Symbol'][i], e)
    chains = _fetch_chains({(tkr, expiry) for tkr, expiry, _ in parsed.values()})
    for i, (tkr, expiry, strike) in parsed.items():
        try:
            optC = chains[(tkr, expiry)]
            opt = optC.puts if opt_type == 'P' else optC.calls
            opt = opt[ (opt['strike'] - strike).abs() < 1 ]
            df.loc[i,'o_Buy'] = pd.Series(opt['bid']).values[0]
            sell = float(df['sell'][i])
            df.loc[i,'o_GainLoss'] = (df['o_Buy'][i] - sell) * df['Quantity'][i]
            df.loc[i,'o_percent'] = round(df['o_GainLoss'][i] * 100 / sell / abs(df['Quantity'][i]))
            df.loc[i, 'o_rate'] = calc_rate(df['o_Buy'][i], df['Symbol'][i])
        except Exception as e:
            print(tkr, e)
    df['o_percent'] = pd.to_numeric(df['o_percent'], errors='coerce').astype('Int64')
    df['o_rate'] = pd.to_numeric(df['o_rate'], errors='coerce').astype('Int64')
    return df[['Account', 'Symbol', 'o_Buy', 'sell', 'o_GainLoss', 'o_percent', 'o_rate']]

In [118]:
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.width', 1000)  # display all columns without wrapping
pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', None)  # display all rows

df = pd.read_csv(r'..\data_save\out_fidelity.csv', on_bad_lines='skip')
IS_OPTION = r'^[A-Za-z.]+\d{6}[P][\d.]+$'
df = df[df['Symbol'].notna() & df['Symbol'].astype(str).str.match(IS_OPTION)] # options only
df = df[~df['Quantity'].isin(['sell', 'buy'])]
print(df[['Account', 'Symbol', 'sell']]) #testing
df_P = _scan(df, 'P')
df_P = df_P.sort_values(by=['o_percent'], ascending=False)
print(df_P)

         Account           Symbol  sell
5      218320886     AGQ260918P81     4
6      X83939133     AGQ260918P74  3.72
7      X83939133     AGQ260918P76   3.2
15     X83939133    APP260918P310  21.3
21     218320886   AVGO260918P360    18
22     218320886   CBRS260918P220    25
24     218320886   CIEN260918P320  15.2
33     X65750304   COIN260918P175   6.4
51     X83939133    CRWV260918P90  7.45
60   Rosetta4260  GOOGL260918P345    14
64    RosettaIRA    LASR260918P50   4.9
80     X83939133   NVDA260914P220     2
81     X83939133   NVDA260916P220   4.8
87     218320886    PPLT260918P16  0.45
94     218320886    RKLB260918P63  2.65
103    231736622    SMCI260918P40  2.44
115    218320886   SPCX260918P150     6
AGQ can't multiply sequence by non-int of type 'numpy.float64'
AGQ can't multiply sequence by non-int of type 'numpy.float64'
AGQ can't multiply sequence by non-int of type 'numpy.float64'
APP can't multiply sequence by non-int of type 'numpy.float64'
AVGO can't multiply sequence

KeyError: 'o_percent'

In [ ]:
pd.set_option('display.width', 1000)  # display all columns without wrapping
df = pd.read_csv(r'..\data_save\out_fidelity.csv', on_bad_lines='skip')
IS_OPTION = r'^[A-Za-z.]+\d{6}[C][\d.]+$'
df = df[df['Symbol'].notna() & df['Symbol'].astype(str).str.match(IS_OPTION)] # options only
df = df[~df['Quantity'].isin(['sell', 'buy'])]
df = df[~df['CostBasisPerShare'].notna()] # exclude long ITM call
df_C = _scan(df, 'C')
df_C = df_C.sort_values(by=['o_percent'], ascending=False)
print(df_C)